# Import Libraries

In [ ]:
import numpy as np
from tqdm.notebook import tqdm
tqdm.pandas()
import pandas as pd
import os
import matplotlib.pyplot as plt

import shutil

from joblib import Parallel, delayed
from IPython.display import display
from ultralytics import YOLO

# Meta Data

In [ ]:
DIM       = 2560
MODEL     = 'yolov8n'
BATCH     = 2
EPOCHS    = 100
OPTMIZER  = 'Adam'

REMOVE_NOBBOX = True # remove images with no bbox
ROOT_DIR  = 'tensorflow-great-barrier-reef'
IMAGE_DIR = 'output/images' # directory to save images
LABEL_DIR = 'output/labels' # directory to save labels

## Create Directories

In [3]:
!mkdir -p {IMAGE_DIR}
!mkdir -p {LABEL_DIR}

## Get Paths

In [ ]:
# Train Data
df = pd.read_csv(f'{ROOT_DIR}/train.csv')
df['old_image_path'] = f'{ROOT_DIR}/train_images/video_'+df.video_id.astype(str)+'/'+df.video_frame.astype(str)+'.jpg'
df['image_path']  = f'{IMAGE_DIR}/'+df.image_id+'.jpg'
df['label_path']  = f'{LABEL_DIR}/'+df.image_id+'.txt'
df['annotations'] = df['annotations'].progress_apply(eval)
display(df.head(2))

## Number of BBoxes
##### Nearly 80% images are without any bbox.

In [ ]:
df['num_bbox'] = df['annotations'].progress_apply(lambda x: len(x))
data = (df.num_bbox>0).value_counts(normalize=True)*100
print(f"No BBox: {data[0]:0.2f}% | With BBox: {data[1]:0.2f}%")

# Clean Data

In [6]:
if REMOVE_NOBBOX:
    df = df.query("num_bbox>0")

#  Write Images

In [7]:
def make_copy(row):
    shutil.copyfile(row.old_image_path, row.image_path)
    return

In [ ]:
image_paths = df.old_image_path.tolist()
_ = Parallel(n_jobs=-1, backend='threading')(delayed(make_copy)(row) for _, row in tqdm(df.iterrows(), total=len(df)))

# Helper

In [ ]:
# COCO [x, y, w, h] → VOC [x1, y1, x2, y2]
def coco_to_voc(boxes):
    boxes = np.asarray(boxes)
    voc_boxes = boxes.copy()
    if boxes.ndim == 1:
        voc_boxes[2] = voc_boxes[0]+voc_boxes[2]
        voc_boxes[3] = voc_boxes[1]+voc_boxes[3]
    else:
        voc_boxes[:,2] = voc_boxes[:,0]+voc_boxes[:,2]
        voc_boxes[:,3] = voc_boxes[:,1]+voc_boxes[:,3]
    return voc_boxes

# VOC -> YOLO [x_center, y_center, w, h], all normalized
def voc_to_yolo(boxes_voc, img_height, img_width):
    boxes = np.asarray(boxes_voc)
    x_min, y_min, x_max, y_max = boxes[:,0], boxes[:,1], boxes[:,2], boxes[:,3]
    x_c = (x_min + x_max)/2. / img_width
    y_c = (y_min + y_max)/2. / img_height
    w   = (x_max-x_min)/img_width
    h   = (y_max-y_min)/img_height
    return np.stack([x_c, y_c, w, h], axis=1)

def clip_boxes(boxes, img_height, img_width):
    boxes = np.asarray(boxes)
    boxes[..., 0] = np.clip(boxes[..., 0], 0, img_width-1)
    boxes[..., 1] = np.clip(boxes[..., 1], 0, img_height-1)
    boxes[..., 2] = np.clip(boxes[..., 2], 0, img_width-1)
    boxes[..., 3] = np.clip(boxes[..., 3], 0, img_height-1)
    return boxes

def str2annot(s):
    lines = s.strip().split('\n')
    if len(lines)==0 or lines[0]=='':
        return np.zeros((0, 5))
    arr = [list(map(float, l.split())) for l in lines if l.strip()]
    return np.array(arr)

def annots_to_str(annots):
    return '\n'.join(' '.join(map(str, row)) for row in annots)+'\n'

def get_bbox(annots):
    bboxes = [list(annot.values()) for annot in annots]
    return bboxes

def get_imgsize(row):
    with Image.open(row['image_path']) as img:
        row['width'], row['height'] = img.size
    return row

np.random.seed(32)
colors = [(np.random.randint(255), np.random.randint(255), np.random.randint(255)) for _ in range(1)]

## Create BBox

In [ ]:
df['bboxes'] = df.annotations.progress_apply(get_bbox)
df.head(2)

## Get Image-Size

In [ ]:
df['width']  = 1280
df['height'] = 720
display(df.head(2))

# Create Labels


In [ ]:
cnt = 0
all_bboxes = []
bboxes_info = []
for row_idx in tqdm(range(df.shape[0])):
    row = df.iloc[row_idx]
    image_height = row['height']
    image_width  = row['width']
    bboxes_coco  = np.array(row['bboxes']).astype(np.float32).copy()
    num_bbox     = len(bboxes_coco)
    names        = ['cots'] * num_bbox
    labels       = np.array([0]*num_bbox)[..., None].astype(str)
    # Create Annotation (YOLO)
    with open(row['label_path'], 'w') as f:
        if num_bbox < 1:
            f.write('')
            cnt += 1
            continue
        bboxes_voc  = coco_to_voc(bboxes_coco)
        bboxes_voc  = clip_boxes(bboxes_voc, image_height, image_width)
        bboxes_yolo = voc_to_yolo(bboxes_voc, image_height, image_width).astype(str)
        all_bboxes.extend(bboxes_yolo.astype(float))
        bboxes_info.extend([[row['image_id'], row['video_id'], row['sequence']]] * len(bboxes_yolo))
        annots = np.concatenate([labels, bboxes_yolo], axis=1)
        string = annots_to_str(annots)
        f.write(string)
print('Missing:', cnt)

# Create Folds

In [ ]:
from sklearn.model_selection import GroupKFold
kf = GroupKFold(n_splits = 3)
df = df.reset_index(drop=True)
df['fold'] = -1
for fold, (train_idx, val_idx) in enumerate(kf.split(df, groups=df.video_id.tolist())):
    df.loc[val_idx, 'fold'] = fold
display(df.fold.value_counts())

# BBox Distribution

In [ ]:
bbox_df = pd.DataFrame(np.concatenate([bboxes_info, all_bboxes], axis=1),
             columns=['image_id','video_id','sequence',
                     'xmid','ymid','w','h'])
bbox_df[['xmid','ymid','w','h']] = bbox_df[['xmid','ymid','w','h']].astype(float)
bbox_df['area'] = bbox_df.w * bbox_df.h * 1280 * 720
bbox_df = bbox_df.merge(df[['image_id','fold']], on='image_id', how='left')
bbox_df.head(2)

## `x_center` Vs `y_center`

In [ ]:
from scipy.stats import gaussian_kde

all_bboxes = np.array(all_bboxes)

x_val = all_bboxes[...,0]
y_val = all_bboxes[...,1]

xy = np.vstack([x_val,y_val])
z = gaussian_kde(xy)(xy)

fig, ax = plt.subplots(figsize = (10, 10))
# ax.axis('off')
ax.scatter(x_val, y_val, c=z, s=100, cmap='viridis')
# ax.set_xlabel('x_mid')
# ax.set_ylabel('y_mid')
plt.show()

## `width` Vs `height`

In [ ]:
x_val = all_bboxes[...,2]
y_val = all_bboxes[...,3]

# Calculate the point density
xy = np.vstack([x_val,y_val])
z = gaussian_kde(xy)(xy)

fig, ax = plt.subplots(figsize = (10, 10))
# ax.axis('off')
ax.scatter(x_val, y_val, c=z, s=100, cmap='viridis')
# ax.set_xlabel('bbox_width')
# ax.set_ylabel('bbox_height')
plt.show()

## Area

In [ ]:
import matplotlib as mpl
import seaborn as sns

f, ax = plt.subplots(figsize=(12, 6))
sns.despine(f)

sns.histplot(
    bbox_df,
    x="area", hue="fold",
    multiple="stack",
    palette="viridis",
    edgecolor=".3",
    linewidth=.5,
    log_scale=True,
)
ax.xaxis.set_major_formatter(mpl.ticker.ScalarFormatter())
ax.set_xticks([500, 1000, 2000, 5000, 10000]);

# Visualization

In [ ]:
import torch
from torchvision.utils import draw_bounding_boxes
from torchvision.transforms.functional import to_tensor, to_pil_image
from PIL import Image

def visualize_yolo(img_path, yolo_boxes, labels, names, img_height, img_width, color=(255,0,0)):
    img = Image.open(img_path).convert('RGB')
    boxes_xyxy = []
    for bx in yolo_boxes:
        x_center, y_center, w, h = bx
        x1 = int((x_center - w/2) * img_width)
        y1 = int((y_center - h/2) * img_height)
        x2 = int((x_center + w/2) * img_width)
        y2 = int((y_center + h/2) * img_height)
        boxes_xyxy.append([x1, y1, x2, y2])
    if len(boxes_xyxy) == 0:
        return img
    boxes = torch.tensor(boxes_xyxy, dtype=torch.int)
    label_names = [names[i] for i in labels]
    img_tensor = (to_tensor(img) * 255).to(torch.uint8)
    if isinstance(color, tuple):
        colors = [color] * len(boxes)
    else:
        colors = color
    img_with_boxes = draw_bounding_boxes(img_tensor, boxes, labels=label_names, colors=colors, width=2)
    return to_pil_image(img_with_boxes)

In [ ]:
df2 = df[(df.num_bbox>0)].sample(100)
y = 3; x = 2
plt.figure(figsize=(12.8*x, 7.2*y))
for idx in range(x*y):
    row = df2.iloc[idx]
    image_height = row.height
    image_width = row.width
    with open(row.label_path) as f:
        annot = str2annot(f.read())
    bboxes_yolo = annot[...,1:]
    labels      = annot[...,0].astype(int).tolist()
    names       = ['cots']*len(bboxes_yolo)
    plt.subplot(y, x, idx+1)
    img_with_boxes = visualize_yolo(
        row.image_path, bboxes_yolo, labels, names, image_height, image_width, color=colors[0]
    )
    plt.imshow(img_with_boxes)
    plt.axis('off')
plt.tight_layout()
plt.show()

# Dataset

In [ ]:
train_files = []
val_files   = []
train_df = df.query("fold!=@FOLD")
valid_df = df.query("fold==@FOLD")
train_files += list(train_df.image_path.unique())
val_files += list(valid_df.image_path.unique())
len(train_files), len(val_files)

# Configuration

In [ ]:
import yaml

cwd = 'output/working'
with open(os.path.join(cwd, 'train.txt'), 'w') as f:
    for path in train_df.image_path.tolist():
        f.write(path + '\n')
with open(os.path.join(cwd, 'val.txt'), 'w') as f:
    for path in valid_df.image_path.tolist():
        f.write(path + '\n')

data = dict(
    path = cwd,
    train = 'train.txt',
    val = 'val.txt',
    nc = 1,
    names = ['cots'],
)
with open(os.path.join(cwd, 'gbr.yaml'), 'w') as outfile:
    yaml.dump(data, outfile, default_flow_style=False)

with open(os.path.join(cwd, 'gbr.yaml'), 'r') as f:
    print('\nyaml:')
    print(f.read())

In [22]:
# YOLOv8 optimized hyperparameters for detection
lr0: 0.01  # initial learning rate
lrf: 0.01  # final learning rate factor
momentum: 0.937  # momentum
weight_decay: 0.0005  # weight decay
warmup_epochs: 3.0  # warmup epochs
warmup_momentum: 0.8  # warmup momentum
warmup_bias_lr: 0.1  # warmup bias lr
box: 7.5  # box loss gain
cls: 0.5  # cls loss gain
dfl: 1.5  # distribution focal loss gain
label_smoothing: 0.0  # label smoothing
nbs: 32  # nominal batch size
hsv_h: 0.04  # hsv hue augmentation
hsv_s: 0.7  # hsv saturation augmentation
hsv_v: 0.5  # hsv value augmentation
degrees: 0.0  # image rotation (+/- deg)
translate: 0.1  # image translation (+/- fraction)
scale: 0.5  # image scale (+/- gain)
shear: 0.0  # image shear (+/- deg)
perspective: 0.0  # image perspective (+/- fraction)
flipud: 0.5  # image flip up-down (probability)
fliplr: 0.5  # image flip left-right (probability)
mosaic: 1.0  # image mosaic (probability)
mixup: 0.1  # image mixup (probability)
copy_paste: 0.3  # segment copy-paste (probability)

# Training

In [23]:
PROJECT = 'output/yolov8'
NAME    = f'gbr_{MODEL}_fold'

In [ ]:
model_path = f'models/{MODEL}.pt'
model = YOLO(model_path)

model.train(data='output/working/gbr.yaml', 
            epochs=EPOCHS, 
            batch=BATCH, 
            imgsz=DIM, 
            optimizer='AdamW', 
            project=PROJECT, 
            name=NAME, 
            exist_ok=True)

## Output Files

In [ ]:
OUTPUT_DIR = f'output/yolov8/{NAME}'
!ls {OUTPUT_DIR}

In [ ]:
best_model_path = f'{OUTPUT_DIR}/weights/best.pt'
model = YOLO(best_model_path)

val_results = model.val(
    data='output/working/gbr.yaml',
    imgsz=DIM,
    batch=BATCH,
    conf=0.25,
    iou=0.45,
    split='val',
    project=PROJECT,     
    name=f'{NAME}_val'     
)

print("\n" + "="*50)
print("Validation Results:")
print("="*50)
print(f"mAP50:    {val_results.box.map50:.4f}")
print(f"Precision: {val_results.box.mp:.4f}")
print(f"Recall:    {val_results.box.mr:.4f}")
print("="*50)

precision = val_results.box.mp
recall = val_results.box.mr
beta = 2
f2_score = (1 + beta**2) * (precision * recall) / (beta**2 * precision + recall) if (beta**2 * precision + recall) > 0 else 0
print(f"F2-Score:  {f2_score:.4f}")
print("="*50)
print(f"\nValidation results saved to: {PROJECT}/{NAME}_val")

# Class Distribution

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'num_bbox' not in df.columns:
    df_original = pd.concat([train_df, valid_df]).reset_index(drop=True)
else:
    df_original = df

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(data=df_original, x='num_bbox', bins=30, kde=True, ax=axes[0])
axes[0].set_title('Overall Distribution of Objects per Image')
axes[0].set_xlabel('Number of COTS per Image')
axes[0].set_ylabel('Number of Images')

sns.histplot(data=df_original, x='num_bbox', hue='fold', bins=30, multiple="stack", ax=axes[1])
axes[1].set_title('Objects per Image by Fold')
axes[1].set_xlabel('Number of COTS per Image')
axes[1].set_ylabel('Number of Images')

sns.boxplot(data=df_original, x='fold', y='num_bbox', ax=axes[2])
axes[2].set_title('Objects Distribution across Folds')
axes[2].set_xlabel('Fold')
axes[2].set_ylabel('Number of COTS per Image')

plt.tight_layout()
plt.show()

# Batch Image

In [ ]:
import matplotlib.pyplot as plt
for i in range(3):
    plt.figure(figsize=(10, 10))
    try:
        plt.imshow(plt.imread(f'{OUTPUT_DIR}/train_batch{i}.jpg'))
        plt.show()
    except FileNotFoundError:
        print(f"train_batch{i}.jpg not found.")

## GT Vs Pred

In [ ]:
fig, ax = plt.subplots(3, 2, figsize=(2*9,3*5), constrained_layout=True)
for row in range(3):
    try:
        ax[row][0].imshow(plt.imread(f'{OUTPUT_DIR}/val_batch{row}_labels.jpg'))
        ax[row][0].set_xticks([])
        ax[row][0].set_yticks([])
        ax[row][0].set_title(f'{OUTPUT_DIR}/val_batch{row}_labels.jpg', fontsize=12)
    except FileNotFoundError:
        ax[row][0].text(0.5, 0.5, 'Labels image not found', ha='center', va='center', transform=ax[row][0].transAxes)
    
    try:
        ax[row][1].imshow(plt.imread(f'{OUTPUT_DIR}/val_batch{row}_pred.jpg'))
        ax[row][1].set_xticks([])
        ax[row][1].set_yticks([])
        ax[row][1].set_title(f'{OUTPUT_DIR}/val_batch{row}_pred.jpg', fontsize=12)
    except FileNotFoundError:
        ax[row][1].text(0.5, 0.5, 'Pred image not found', ha='center', va='center', transform=ax[row][1].transAxes)
plt.show()

# Result

## Score Vs Epoch

In [ ]:
import pandas as pd
results_csv = f'{OUTPUT_DIR}/results.csv'
if os.path.exists(results_csv):
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    plt.figure(figsize=(30,15))
    if 'epoch' in df.columns and 'train/box_loss' in df.columns:
        plt.plot(df['epoch'], df['train/box_loss'], label='Train Box Loss')
        plt.plot(df['epoch'], df['val/box_loss'], label='Val Box Loss')
        plt.plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP50-95')
        plt.legend()
        plt.title('Training Results')
        plt.show()
    else:
        print("Expected columns not found in results.csv. Check the file structure.")
else:
    print("results.csv not found. Run training first.")

## Confusion Matrix

In [ ]:
plt.figure(figsize=(12,10))
plt.axis('off')
try:
    plt.imshow(plt.imread(f'{OUTPUT_DIR}/confusion_matrix.png'));
except FileNotFoundError:
    print("confusion_matrix.png not found in YOLOv8 output.")

## Metrics

In [ ]:
for metric in ['BoxF1', 'BoxPR', 'BoxP', 'BoxR']:
    print(f'Metric: {metric}')
    plt.figure(figsize=(12,10))
    plt.axis('off')
    try:
        plt.imshow(plt.imread(f'{OUTPUT_DIR}/{metric}_curve.png'));
        plt.show()
    except FileNotFoundError:
        print(f"{metric}_curve.png not found.")